In [2]:
import sys
sys.path.append("/Users/srishtikashyap/Desktop/Pyspark-Practice/src")

from spark_session import create_spark_session

spark = create_spark_session()

print(spark.sparkContext.appName)

IBM_HR_Analytics_Project


In [3]:
import sys
sys.path.append("/Users/srishtikashyap/Desktop/Pyspark-Practice/src")

from data_loader import load_hr_data

df = load_hr_data(spark, "/Users/srishtikashyap/Desktop/Pyspark-Practice/Data/data.csv")

df.show(5)
print("Rows:", df.count())

26/09/30 23:07:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+---------+-----------------+---------+--------------------+----------------+---------+--------------+-------------+--------------+-----------------------+------+----------+--------------+--------+--------------------+---------------+-------------+-------------+-----------+------------------+------+--------+-----------------+-----------------+------------------------+-------------+----------------+-----------------+---------------------+---------------+--------------+------------------+-----------------------+--------------------+
|Age|Attrition|   BusinessTravel|DailyRate|          Department|DistanceFromHome|Education|EducationField|EmployeeCount|EmployeeNumber|EnvironmentSatisfaction|Gender|HourlyRate|JobInvolvement|JobLevel|             JobRole|JobSatisfaction|MaritalStatus|MonthlyIncome|MonthlyRate|NumCompaniesWorked|Over18|OverTime|PercentSalaryHike|PerformanceRating|RelationshipSatisfaction|StandardHours|StockOptionLevel|TotalWorkingYears|TrainingTimesLastYear|WorkLifeBalanc

In [4]:
import sys
sys.path.append("/Users/srishtikashyap/Desktop/Pyspark-Practice/src")

from cleaning import clean_hr_data

cleaned_df = clean_hr_data(df)

cleaned_df.printSchema()
print("Rows:", cleaned_df.count())

root
 |-- Age: integer (nullable = true)
 |-- Attrition: string (nullable = true)
 |-- BusinessTravel: string (nullable = true)
 |-- DailyRate: integer (nullable = true)
 |-- Department: string (nullable = true)
 |-- DistanceFromHome: integer (nullable = true)
 |-- Education: integer (nullable = true)
 |-- EducationField: string (nullable = true)
 |-- EmployeeNumber: integer (nullable = true)
 |-- EnvironmentSatisfaction: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- HourlyRate: integer (nullable = true)
 |-- JobInvolvement: integer (nullable = true)
 |-- JobLevel: integer (nullable = true)
 |-- JobRole: string (nullable = true)
 |-- JobSatisfaction: integer (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Monthly_Income: integer (nullable = true)
 |-- MonthlyRate: integer (nullable = true)
 |-- NumCompaniesWorked: integer (nullable = true)
 |-- OverTime: string (nullable = true)
 |-- PercentSalaryHike: integer (nullable = true)
 |-- Performanc

In [5]:
import sys
sys.path.append("/Users/srishtikashyap/Desktop/Pyspark-Practice/src")

from transformations import add_hr_features

final_df = add_hr_features(cleaned_df)

final_df.select(
    "Age",
    "Age_Group",
    "Attrition",
    "Attrition_Flag"
).show(10)

+---+---------+---------+--------------+
|Age|Age_Group|Attrition|Attrition_Flag|
+---+---------+---------+--------------+
| 27|    25-34|       No|             0|
| 56|      55+|       No|             0|
| 53|    45-54|       No|             0|
| 41|    35-44|       No|             0|
| 37|    35-44|      Yes|             1|
| 32|    25-34|       No|             0|
| 48|    45-54|       No|             0|
| 36|    35-44|       No|             0|
| 31|    25-34|      Yes|             1|
| 27|    25-34|      Yes|             1|
+---+---------+---------+--------------+
only showing top 10 rows


In [6]:
print("Rows:", final_df.count())
print("Columns:", len(final_df.columns))

Rows: 1470
Columns: 34


In [7]:
import sys
sys.path.append("/Users/srishtikashyap/Desktop/Pyspark-Practice/src")

from validation import validate_hr_data

validation_results = validate_hr_data(final_df)

Total rows: 1470
Duplicate rows: 0

Null values:
No null values found.


In [8]:
import sys
sys.path.append("/Users/srishtikashyap/Desktop/Pyspark-Practice/src")

from pipeline import run_pipeline

final_df, validation_results = run_pipeline(
    spark,
    "/Users/srishtikashyap/Desktop/Pyspark-Practice/Data/data.csv"
)

final_df.show(5)

Total rows: 1470
Duplicate rows: 0

Null values:
No null values found.
+---+---------+--------------+---------+--------------------+----------------+---------+--------------+--------------+-----------------------+------+----------+--------------+--------+--------------------+---------------+-------------+--------------+-----------+------------------+--------+-----------------+-----------------+------------------------+----------------+-----------------+---------------------+---------------+--------------+------------------+-----------------------+--------------------+--------------+---------+
|Age|Attrition|BusinessTravel|DailyRate|          Department|DistanceFromHome|Education|EducationField|EmployeeNumber|EnvironmentSatisfaction|Gender|HourlyRate|JobInvolvement|JobLevel|             JobRole|JobSatisfaction|MaritalStatus|Monthly_Income|MonthlyRate|NumCompaniesWorked|OverTime|PercentSalaryHike|PerformanceRating|RelationshipSatisfaction|StockOptionLevel|TotalWorkingYears|TrainingTimesL

In [9]:
output_path = "/Users/srishtikashyap/Desktop/Pyspark-Practice/Data/processed/hr_data"

final_df.write \
    .mode("overwrite") \
    .parquet(output_path)

In [10]:
parquet_df = spark.read.parquet(output_path)

parquet_df.show(5)
parquet_df.printSchema()

+---+---------+--------------+---------+--------------------+----------------+---------+--------------+--------------+-----------------------+------+----------+--------------+--------+--------------------+---------------+-------------+--------------+-----------+------------------+--------+-----------------+-----------------+------------------------+----------------+-----------------+---------------------+---------------+--------------+------------------+-----------------------+--------------------+--------------+---------+
|Age|Attrition|BusinessTravel|DailyRate|          Department|DistanceFromHome|Education|EducationField|EmployeeNumber|EnvironmentSatisfaction|Gender|HourlyRate|JobInvolvement|JobLevel|             JobRole|JobSatisfaction|MaritalStatus|Monthly_Income|MonthlyRate|NumCompaniesWorked|OverTime|PercentSalaryHike|PerformanceRating|RelationshipSatisfaction|StockOptionLevel|TotalWorkingYears|TrainingTimesLastYear|WorkLifeBalance|YearsAtCompany|YearsInCurrentRole|YearsSinceLas

In [11]:
print("Original rows:", final_df.count())
print("Parquet rows:", parquet_df.count())

print("Original columns:", len(final_df.columns))
print("Parquet columns:", len(parquet_df.columns))

Original rows: 1470
Parquet rows: 1470
Original columns: 34
Parquet columns: 34
